# Board evaluation

In [35]:
import torch
import pandas as pd
import os
import chess
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

DATASET_DIR = os.path.join("..", "dataset")
MAIN_DATASET = os.path.join(DATASET_DIR, "chessData.csv")
df = pd.read_csv(MAIN_DATASET)

df["Evaluation"] = pd.to_numeric(df["Evaluation"], downcast="integer", errors="coerce")
df = df.dropna(subset=["Evaluation"]).reset_index(drop=True)

MODEL_PATH = os.path.join("..", "JohnBordello_eval.pt")

In [ ]:
class ChessDataset(Dataset):
    def __init__(self, df):
        self.fens = df["FEN"].tolist()
        self.evals = df["Evaluation"].astype(str).tolist()

    def __len__(self):
        return len(self.fens)

    def __getitem__(self, idx):
        fen = self.fens[idx]
        eval_raw = self.evals[idx]
        
        if eval_raw.startswith('#'):
            mate_val = int(eval_raw.replace('#', ''))
            if mate_val > 0:
                eval_score = 10000.0 - (mate_val * 10)
            else:
                eval_score = -10000.0 - (mate_val * 10)
        else:
            eval_score = float(eval_raw)    
        target = torch.tensor([eval_score / 100.0], dtype=torch.float32)
        board = fast_fen_to_tensor(fen)
        split = fen.split(" ")
        active_player = 1 if split[1] == 'w' else 0
        halfmove = int(split[5])
        return {
            "board_tensor": board,
            "evaluation": target,
            "active_player": torch.tensor(active_player, dtype=torch.long),
            "halfmove": torch.tensor(halfmove, dtype=torch.float32),    
        }
    
dataset = ChessDataset(df)
loader = DataLoader(
    dataset, 
    batch_size=512,
    shuffle=True, 
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

In [ ]:
def get_en_passant(fen):
    matrix = torch.zeros((8,8), dtype=torch.float32)
    fen_en_passant = fen.split(" ")[3]
    if fen_en_passant != "-":
        square = chess.parse_square(fen_en_passant)
        matrix[square % 8][square % 8] = 1
    return matrix

def get_piece_matrix(board, piece_type, color):
    matrix = torch.zeros((8,8), dtype=torch.float32)
    for sq in board.pieces(piece_type, color):
        r = 7 - chess.square_rank(sq)
        c = chess.square_file(sq)
        matrix[r, c] = 1
    return matrix

def fast_fen_to_tensor(fen):
    tensor = torch.zeros((13, 8, 8), dtype=torch.float32)
    parts = fen.split(" ")
    board_part = parts[0]
    ep_part = parts[3]
    piece_map = {
        'p': 1, 'r': 2, 'n': 3, 'b': 4, 'q': 5, 'k': 6,
        'P': 7, 'R': 8, 'N': 9, 'B': 10, 'Q': 11, 'K': 12
    }
    row, col = 0, 0
    for char in board_part:
        if char == '/':
            row += 1
            col = 0
        elif char.isdigit():
            col += int(char)
        else:
            tensor[piece_map[char], row, col] = 1.0
            col += 1
    if ep_part != '-':
        file_idx = ord(ep_part[0]) - ord('a')
        rank_idx = 8 - int(ep_part[1])
        tensor[0, rank_idx, file_idx] = 1.0
    return tensor

In [13]:
class ConditionalBatchNorm2d(nn.Module):
    def __init__(self, num_features, num_conditions):
        super(ConditionalBatchNorm2d, self).__init__()
        self.num_features = num_features
        self.bn = nn.BatchNorm2d(num_features, affine=False)
        self.gamma = nn.Embedding(num_conditions, num_features)
        self.beta = nn.Embedding(num_conditions, num_features)
        nn.init.ones_(self.gamma.weight)
        nn.init.zeros_(self.beta.weight)

    def forward(self, x, condition):
        out = self.bn(x)
        gamma = self.gamma(condition).unsqueeze(-1).unsqueeze(-1)
        beta = self.beta(condition).unsqueeze(-1).unsqueeze(-1)
        return gamma * out + beta

class JohnBordello(nn.Module):
    def __init__(self, num_piece_channels=13, num_classes=1, num_conditions=2):
        super(JohnBordello, self).__init__()
        self.conv1 = nn.Conv2d(num_piece_channels, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.cbn1 = ConditionalBatchNorm2d(64, num_conditions)
        self.cbn2 = ConditionalBatchNorm2d(128, num_conditions)
        self.cbn3 = ConditionalBatchNorm2d(256, num_conditions)
        self.fc1 = nn.Linear(256, 1024)
        self.fc2 = nn.Linear(1024 + 1, num_classes)

    def forward(self, board_tensor, active_player, halfmove_clock):
        x = self.conv1(board_tensor)
        x = self.cbn1(x, active_player)
        x = F.relu(x)
        x = self.conv2(x)
        x = self.cbn2(x, active_player)
        x = F.relu(x)
        x = self.conv3(x)
        x = self.cbn3(x, active_player)
        x = F.relu(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        halfmove_clock = halfmove_clock.float()
        x = torch.cat([x, halfmove_clock.unsqueeze(1)], dim=1)
        output = self.fc2(x)
        return output


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
n_gpus = torch.cuda.device_count()
print("number of gpus:", n_gpus)
model = JohnBordello()
if n_gpus > 1:
    model = nn.DataParallel(model)
model = model.to(device)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4) 
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
patience = 15
epochs_no_improve = 0
best_loss = float("inf")

for epoch in range(200):
    model.train()
    epoch_loss = 0
    for data in loader:
        board = data["board_tensor"].to(device, non_blocking=True)
        player = data["active_player"].to(device, non_blocking=True)
        halfmove = data["halfmove"].to(device, non_blocking=True)
        target = data["evaluation"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if scaler:
            with torch.amp.autocast('cuda'):
                predictions = model(board, player, halfmove)
                loss = criterion(predictions, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            predictions = model(board, player, halfmove)
            loss = criterion(predictions, target)
            loss.backward()
            optimizer.step()
        epoch_loss += loss.item() * board.size(0)
    epoch_loss /= len(loader.dataset)
    print(f"epoch: {epoch}, loss: {epoch_loss:.4f}")
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "/kaggle/working/JohnBordello_eval.pt")
    else:
        epochs_no_improve += 1
    if epochs_no_improve >= patience:
        print("max patience exceded")
        break

cpu
number of gpus: 0


In [34]:
model = JohnBordello().to(device)
state_dict = torch.load(MODEL_PATH, map_location=device, weights_only=True)
new_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("module."):
        new_state_dict[k[7:]] = v
    else:
        new_state_dict[k] = v
model.load_state_dict(new_state_dict)
model.eval()
def evaluate_board(fen):
    board = fen_to_tensor(fen).unsqueeze(0).to(device)
    split = fen.split(" ")
    active_player_str = split[1]
    active_player_val = 1 if active_player_str == 'w' else 0
    active_player = torch.tensor([active_player_val], dtype=torch.long).to(device)
    halfmove_val = int(split[5])
    halfmove = torch.tensor([halfmove_val], dtype=torch.float32).to(device)
    with torch.no_grad():
        evaluation = model(board, active_player, halfmove)
    return evaluation.item() * 100.0

evaluate_board("rnbqkb1r/pppn1ppp/4p3/3pP3/3P4/8/PPPN1PPP/R1BQKBNR w KQkq - 1 5") #+86

190.2030110359192